# Clasificación con árboles de decisión y bosques aleatorios

En esta práctica se utiliza el conjunto de datos Iris para entrenar dos modelos de clasificación: un árbol de decisión y un bosque aleatorio. Al final se comparan sus resultados mediante precisión, matrices de confusión y reportes de clasificación.

## 1. Importar librerías y cargar los datos

Iris contiene medidas de flores de tres especies. Las cuatro medidas se utilizan como características y la especie es la clase que se desea predecir.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="especie")

print("Primeros registros:")
display(X.head())
print("Cantidad de registros:", X.shape[0])
print("Clases:", iris.target_names)

## 2. Dividir los datos

Se separa el 70% de los registros para entrenamiento y el 30% para prueba. Se utiliza stratify para conservar la proporción de las tres especies en ambos grupos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

## 3. Entrenar los modelos

El árbol se limita a una profundidad de tres niveles para mantenerlo sencillo. El bosque aleatorio se construye con 100 árboles.

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)

bosque = RandomForestClassifier(n_estimators=100, random_state=42)
bosque.fit(X_train, y_train)

## 4. Realizar predicciones y evaluar

Se calcula la precisión y se genera un reporte con precision, recall y F1-score para cada especie.

In [ ]:
pred_arbol = arbol.predict(X_test)
pred_bosque = bosque.predict(X_test)

precision_arbol = accuracy_score(y_test, pred_arbol)
precision_bosque = accuracy_score(y_test, pred_bosque)

print("Precisión del árbol:", round(precision_arbol, 4))
print("Precisión del bosque aleatorio:", round(precision_bosque, 4))

print("\nReporte del árbol de decisión:")
print(classification_report(y_test, pred_arbol, target_names=iris.target_names))

print("\nReporte del bosque aleatorio:")
print(classification_report(y_test, pred_bosque, target_names=iris.target_names))

## 5. Matrices de confusión

Las matrices permiten observar cuántas flores fueron clasificadas correctamente y en cuáles se equivocó cada modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_arbol = confusion_matrix(y_test, pred_arbol)
cm_bosque = confusion_matrix(y_test, pred_bosque)

sns.heatmap(cm_arbol, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names, ax=axes[0])
axes[0].set_title("Árbol de decisión")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Valor real")

sns.heatmap(cm_bosque, annot=True, fmt="d", cmap="Greens",
            xticklabels=iris.target_names, yticklabels=iris.target_names, ax=axes[1])
axes[1].set_title("Bosque aleatorio")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Valor real")

plt.tight_layout()
plt.show()

## 6. Visualizar el árbol de decisión

La gráfica muestra las condiciones que utiliza el árbol para separar las especies.

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(arbol, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, rounded=True)
plt.title("Árbol de decisión para clasificar flores Iris")
plt.show()

## 7. Importancia de las características

Random Forest permite estimar qué características fueron más útiles para realizar las predicciones.

In [ ]:
importancias = pd.Series(bosque.feature_importances_,
                           index=iris.feature_names).sort_values(ascending=False)
print(importancias)

importancias.sort_values().plot(kind="barh", figsize=(8, 4))
plt.title("Importancia de las características - Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Característica")
plt.tight_layout()
plt.show()

## 8. Comparación final

Se muestran las precisiones de ambos modelos para facilitar su comparación. El resultado se obtiene al ejecutar el notebook.

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["Árbol de decisión", "Bosque aleatorio"],
    "Precisión": [precision_arbol, precision_bosque]
})
display(resultados)